# 实践 2.1  VCF解析与基本基因型质控

《统计遗传与智能育种学》第二章 · 电子资源 `02_vcf_qc.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：02_vcf_qc.ipynb。实践使用内嵌的VCF片段示例，解析GT字段并转换为0/1/2剂量矩阵，计算每个位点的缺失率、替代等位基因频率、MAF和观察杂合度，然后按照可修改阈值筛选位点。

核心输出字段：SNP, missing_rate（缺失率）, ALT_AF（替代等位基因频率，alternate allele frequency）, MAF, observed_het（观察杂合度）  
示例过滤：missing_rate <= 0.30 且 MAF >= 0.08

## 输入、完整代码、实际输出与解释

输入与可复现条件：内嵌5个样本、4个SNP的VCF片段；SEED=201；缺失GT按NaN（非数值，not a number）处理。筛选阈值为missing_rate≤0.30且MAF≥0.08。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED=201
rng=np.random.default_rng(SEED)
print('Python',sys.version.split()[0])
print('Platform',platform.platform())
print('numpy',np.__version__,'pandas',pd.__version__,'scikit-learn',sklearn.__version__)
print('SEED',SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 201


In [2]:
from io import StringIO
vcf='''#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\tFORMAT\tS1\tS2\tS3\tS4\tS5\n1\t100\trs1\tA\tG\t60\tPASS\t.\tGT\t0/0\t0/1\t1/1\t0/1\t./.\n1\t220\trs2\tC\tT\t50\tPASS\t.\tGT\t0/0\t0/0\t0/1\t0/0\t0/0\n1\t350\trs3\tG\tA\t55\tPASS\t.\tGT\t1/1\t1/1\t1/1\t0/1\t1/1\n1\t490\trs4\tT\tC\t45\tPASS\t.\tGT\t0/1\t./.\t0/1\t./.\t0/0\n'''
raw=pd.read_csv(StringIO(vcf),sep='\t')
samples=[c for c in raw.columns if c.startswith('S')]
def gt_dose(x):
    if x in {'./.','.|.'}: return np.nan
    a=x.replace('|','/').split('/'); return float(sum(int(v) for v in a))
G=raw[samples].map(gt_dose).T; G.columns=raw.ID; print(G)

rows=[]
for snp in G.columns:
 x=G[snp].to_numpy(float); ok=~np.isnan(x); af=x[ok].sum()/(2*ok.sum()); maf=min(af,1-af); miss=1-ok.mean(); het=np.mean(x[ok]==1)
 rows.append((snp,miss,af,maf,het))
qc=pd.DataFrame(rows,columns=['SNP','missing_rate','ALT_AF','MAF','observed_het'])
max_missing=.30; min_maf=.08; qc['keep']=(qc.missing_rate<=max_missing)&(qc.MAF>=min_maf)
print(qc.round(3).to_string(index=False)); print('kept:',qc.loc[qc.keep,'SNP'].tolist())


ID  rs1  rs2  rs3  rs4
S1  0.0  0.0  2.0  1.0
S2  1.0  0.0  2.0  NaN
S3  2.0  1.0  2.0  1.0
S4  1.0  0.0  1.0  NaN
S5  NaN  0.0  2.0  0.0
SNP  missing_rate  ALT_AF   MAF  observed_het  keep
rs1           0.2   0.500 0.500         0.500  True
rs2           0.0   0.100 0.100         0.200  True
rs3           0.0   0.900 0.100         0.200  True
rs4           0.4   0.333 0.333         0.667 False
kept: ['rs1', 'rs2', 'rs3']


结果解释：rs4因缺失率0.40被删除，rs1–rs3保留；ALT allele frequency与MAF不是同义词，MAF取较小的等位基因频率。观察杂合度是已分型样本的杂合比例，不能替代Hardy–Weinberg检验或测序质控。

可修改练习：把缺失率阈值改为0.50、MAF阈值改为0.15，记录保留位点变化，并解释阈值为什么必须结合样本量与研究目的。